# Comparar cinco modelos cinéticos de flotación con la misma evidencia

**Pregunta:** ¿Qué ajustes cinéticos conservan la respuesta batch y la recuperación del banco al cambiar la residencia?

Nivel: 3. Requisitos previos: Niveles 1 y 2 de este tema. Tiempo de lectura: 24 minutos.



## Objetivos

Comparar cinco formas cinéticas citadas sobre la misma evidencia batch y evaluar por separado la proyección de banco frente a la referencia exacta con tasas distribuidas del motor.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Para la lección de nivel 3 sobre 02-flotation, registra las versiones fijadas antes de ejecutar el análisis; así se puede reconstruir el entorno que produjo cada tabla y figura.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. pipeline.cases.catalog aporta el circuito de cobre y molibdeno. El motor calcula alimentación rougher, tasas y residencia; tiempos batch y ajustes numéricos provienen de engine/data/constants.json. Los registros nominales, de banco corto y dosis modificada se generan cuando la lección los usa. Datos virtuales y referencia exacta comparten supuestos del simulador, sin ser mediciones de laboratorio. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

Las formas candidatas son primer orden, Kelsall rápida/lenta, Klimpel de tasas uniformes, gamma y exponencial estirada. Cada una resume de modo distinto tasas heterogéneas. OreFlow evalúa la misma curva batch virtual y compara proyecciones de banco; una curva ajustada no identifica un mecanismo único.

## Comparación

La cinética de primer orden ofrece una referencia de tasa única. Kelsall (1961) propone fracciones rápida y lenta. Kelsall y las formas de tasas distribuidas representan mejor la heterogeneidad con más parámetros. OreFlow implementa cinco formas en `pipeline/engine/kinetics.py` y mide tanto error batch como error de proyección frente a su distribución exacta de tasas. Es una comparación controlada, no superioridad demostrada con datos de planta ni un algoritmo nuevo.

Resolver el circuito nominal y ajustar nuevamente las cinco formas a observaciones batch virtuales comunes. No se carga una tabla guardada de puntuaciones. El banco exacto de tasas distribuidas constituye la referencia del simulador para comparar los modelos.

In [ ]:
# Resolver el circuito nominal y ajustar nuevamente las cinco formas a observaciones batch
# virtuales comunes
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.kinetics import kinetic_record

case = CASE_BY_ID["copper_molybdenum"]
result = simulate(case.ore, case.plant, case.nominal)
kinetics = kinetic_record(
    result.flotation, result.ore, case.nominal.rougher_cells, result.metrics["rougher_recovery_pct"]
)
models = kinetics["models"]
print("Modelos ajustados:", len(models))


Comparar en paralelo el ajuste batch y error de banco; sus ordenamientos pueden diferir.

In [ ]:
# Comparar en paralelo el ajuste batch y error de banco; sus ordenamientos pueden diferir.
scores = pd.DataFrame(
    [
        {
            "modelo": {
                "first_order": "Primer orden",
                "kelsall": "Kelsall",
                "klimpel": "Klimpel",
                "gamma": "Gamma",
                "stretched_exponential": "Exponencial estirada",
            }.get(m["id"], m["id"]),
            "RMSE batch (puntos)": m["rmse_pct"],
            "predicción del banco (%)": m["bank_projection_pct"],
            "error de banco (puntos)": m["lumping_error_pct"],
            "brecha de extrapolación (puntos)": m["ultimate_gap_pct"],
        }
        for m in models
    ]
)
display(scores.set_index("modelo").round(3).sort_values("RMSE batch (puntos)"))


Graficar los modelos ajustados sobre las mismas observaciones batch. La figura siguiente compara sus proyecciones de banco con la referencia exacta del motor.

In [ ]:
# Graficar los modelos ajustados sobre las mismas observaciones batch.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.scatter(
    kinetics["times_min"],
    kinetics["batch_recovery_pct"],
    color="black",
    zorder=4,
    label="Observaciones virtuales",
)
for m in models:
    ax.plot(
        kinetics["dense_times_min"],
        m["dense_pct"],
        label={
            "first_order": "Primer orden",
            "kelsall": "Kelsall",
            "klimpel": "Klimpel",
            "gamma": "Gamma",
            "stretched_exponential": "Exponencial estirada",
        }.get(m["id"], m["id"]),
    )
ax.set(
    xlabel="Tiempo batch (min)",
    ylabel="Recuperación (%)",
    title="Modelos cinéticos ajustados a datos\nbatch virtuales",
)
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.72)


Las medidas pareadas distinguen interpolación temporal batch y proyección a un banco con residencias mezcladas. Un buen orden batch puede no conservarse en el banco. La referencia exacta existe porque es un experimento de simulador; en laboratorio exigiría caracterización cinética independiente. No interpretar el mejor ajuste virtual como modelo universalmente preferido de flotación.

Comparar las proyecciones de banco con el resultado exacto de tasas distribuidas. Un buen ajuste batch no garantiza una buena proyección por la distribución continua de residencia.

In [ ]:
# Comparar las proyecciones de banco con el resultado exacto de tasas distribuidas.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.barh([m["id"] for m in models], [m["bank_projection_pct"] for m in models])
ax.axhline(
    kinetics["bank"]["exact_true_flotation_pct"],
    color="firebrick",
    linestyle="--",
    label="Referencia exacta",
)
ax.set(ylabel="Recuperación del\nbanco (%)", title="Proyección a celdas continuas")
ax.tick_params(axis="x", rotation=0)
ax.legend(ncol=2, **legend_style)
ax.grid(axis="y", alpha=0.25)
ax.legend(["RMSE batch", "|Error banco|"], fontsize=18, ncol=2, **legend_style)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)


La proyección usa tasas ajustadas con residencia mezclada, en vez de repetir tiempo batch. La separación alrededor de la referencia exacta expone sesgo oculto por el ajuste temporal. Acertar en este banco no valida otro mineral o geometría. Conservar supuestos de residencia al transferir una forma cinética a otro cálculo de proceso.

Recalcular el rougher de cobre y molibdeno nominal y ajustar su respuesta batch virtual con la rutina cinética de producción. La referencia exacta es la distribución de tasas por clase que entrega el circuito, no una curva de laboratorio independiente. Cada modelo ve los mismos tiempos y etiquetas de recuperación, mientras su proyección a un banco continuo prueba otro supuesto de residencia.

In [ ]:
# Recalcular el rougher de cobre y molibdeno nominal y ajustar su respuesta batch virtual con la
# rutina cinética de producción.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.kinetics import kinetic_record

case = CASE_BY_ID["copper_molybdenum"]
nominal = simulate(case.ore, case.plant, case.nominal)
fits = kinetic_record(
    nominal.flotation,
    nominal.ore,
    case.nominal.rougher_cells,
    nominal.metrics["rougher_recovery_pct"],
)
display(
    pd.DataFrame(
        [
            {
                "Modelo": {
                    "first_order": "Primer orden",
                    "kelsall": "Kelsall",
                    "klimpel": "Klimpel",
                    "gamma": "Gamma",
                    "stretched_exponential": "Exponencial estirada",
                }.get(m["id"], m["id"]),
                "RMSE batch (pp)": m["rmse_pct"],
                "Error de banco (pp)": m["lumping_error_pct"],
            }
            for m in fits["models"]
        ]
    )
)


Reducir en una la cantidad de celdas continuas sin cambiar alimentación ni dosis de reactivo. Así cambia la residencia total del banco con una interpretación física clara. Una asíntota batch no predice directamente esa recuperación: la distribución de residencia del tanque incluye material que sale temprano, a diferencia del batch donde todas las partículas comparten la duración de exposición.

In [ ]:
# Reducir en una la cantidad de celdas continuas sin cambiar alimentación ni dosis de reactivo.
short_point = case.nominal.with_values(rougher_cells=max(1, case.nominal.rougher_cells - 1))
short = simulate(case.ore, case.plant, short_point)
print("Recuperación del banco corto (%):", round(short.metrics["rougher_recovery_pct"], 3))


Ajustar la misma familia cinética al banco acortado y proyectar cada ajuste sobre su geometría. Mantener separados los dos registros para que el cambio de residencia no sobrescriba la referencia comparativa. Comparar el error con el resultado exacto de tasas distribuidas de cada banco. Un residuo batch pequeño puede coexistir con una proyección sesgada porque las observaciones de exposición finita no identifican toda la distribución de tasas.

In [ ]:
# Ajustar la misma familia cinética al banco acortado y proyectar cada ajuste sobre su geometría.
short_fits = kinetic_record(
    short.flotation, short.ore, short_point.rougher_cells, short.metrics["rougher_recovery_pct"]
)
print(
    "Recuperación exacta del banco corto (%):",
    round(short_fits["bank"]["exact_true_flotation_pct"], 3),
)


Aumentar diez por ciento la dosis de colector y volver a ejecutar el circuito completo. La perturbación modifica la cinética de colección, no la alimentación fresca. Puede cambiar la colección de ganga además de la recuperación valiosa, por lo que aumentar recuperación rougher no implica mejorar calidad de concentrado. Distinguir esta respuesta química condicional del cambio de cantidad de tanques.

In [ ]:
# Aumentar diez por ciento la dosis de colector y volver a ejecutar el circuito completo.
dose_point = case.nominal.with_values(collector_gpt=case.nominal.collector_gpt * 1.1)
dosed = simulate(case.ore, case.plant, dose_point)
print("Recuperación con dosis modificada (%):", round(dosed.metrics["rougher_recovery_pct"], 3))


Proyectar las formas ajustadas con la nueva condición de colector y conservar, para cada forma, el error batch y el error de banco. Las aproximaciones de primer orden, rápida/lenta y tasas distribuidas pueden cambiar de orden entre ajuste batch y proyección al banco. Se pregunta por error de aproximación dentro de un simulador, no por el modelo más preciso para todas las plantas. Los parámetros en sus límites permanecen en los diagnósticos y deben inspeccionarse antes de extrapolar.

In [ ]:
# Proyectar las formas ajustadas con la nueva condición de colector y conservar, para cada forma,
# el error batch y el error de banco.
dose_fits = kinetic_record(
    dosed.flotation, dosed.ore, dose_point.rougher_cells, dosed.metrics["rougher_recovery_pct"]
)
errors = pd.DataFrame(
    [
        {
            "Modelo": {
                "first_order": "Primer orden",
                "kelsall": "Kelsall",
                "klimpel": "Klimpel",
                "gamma": "Gamma",
                "stretched_exponential": "Exponencial estirada",
            }.get(m["id"], m["id"]),
            "RMSE batch (pp)": m["rmse_pct"],
            "Error de banco (pp)": abs(m["lumping_error_pct"]),
        }
        for m in dose_fits["models"]
    ]
)
display(errors.round(4))


Comparar errores de ajuste batch y proyección al banco continuo con el cambio controlado de colector.

In [ ]:
# Comparar errores de ajuste batch y proyección al banco continuo con el cambio controlado de
# colector..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
errors.set_index("Modelo").plot.barh(ax=ax)
ax.set(xlabel="Error de\nrecuperación\n(pp)", ylabel="", title="Colector +10%: ajuste y banco")
ax.tick_params(axis="x", rotation=0)
ax.legend(["RMSE batch", "|Error banco|"], fontsize=18, ncol=2, **legend_style)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)


Las barras pareadas muestran que ajustar una curva batch y proyectar un banco son tareas distintas. Se grafica el error absoluto de banco para que un sesgo negativo no parezca beneficio; la tabla conserva identidades y magnitudes. Ambos errores usan puntos porcentuales. Un error pequeño mide la compresión de la distribución de tasas del motor y no informa sobre muestreo de laboratorio, calibración de espuma ni desempeño en una planta productiva.

Aquí se reajustan las formas cinéticas en lugar de confiar en puntuaciones guardadas. El banco recién resuelto define la distribución común y cada aproximación recibe los mismos tiempos virtuales. Comparar errores de proyección mide qué se pierde al comprimir tasas en cada forma paramétrica. No se agrega experimento descargado ni medición inventada; la referencia exacta sigue siendo del simulador.

Comparar cada proyección después de acortar el banco. Se separan reajuste de la respuesta virtual modificada y transferencia de parámetros nominales congelados a otra residencia. Solo la segunda es transferencia con parámetros fijos. Ambas usan la referencia exacta del banco corto; los errores con signo distinguen sobrepredicción y subpredicción.

In [ ]:
# Comparar cada proyección después de acortar el banco.
from pipeline.engine.kinetics import MODELS

nominal_by_id = {m["id"]: m for m in fits["models"]}
short_by_id = {m["id"]: m for m in short_fits["models"]}
short_exact = short_fits["bank"]["exact_true_flotation_pct"]
residence_rows = []
for model in MODELS:
    nominal_fit = nominal_by_id[model.id]
    refit = short_by_id[model.id]
    q = [
        nominal_fit["parameters"][name] / (100.0 if name in ("R_inf", "phi") else 1.0)
        for name in model.names
    ]
    frozen = 100.0 * model.project(
        q, short_point.rougher_cells, short_fits["bank"]["cell_residence_min"]
    )
    residence_rows.append(
        {
            "Modelo": model.id,
            "Banco nominal (%)": nominal_fit["bank_projection_pct"],
            "Banco corto reajustado (%)": refit["bank_projection_pct"],
            "Banco corto congelado (%)": frozen,
            "Banco corto exacto (%)": short_exact,
            "Error con signo de reajuste (pp)": refit["lumping_error_pct"],
            "Error con signo congelado (pp)": frozen - short_exact,
            "Error absoluto de reajuste (pp)": abs(refit["lumping_error_pct"]),
            "Error absoluto congelado (pp)": abs(frozen - short_exact),
        }
    )
residence = pd.DataFrame(residence_rows)
display(residence.round(4))


Graficar errores con signo del banco corto para representaciones reajustadas y congeladas de las cinco formas.

In [ ]:
# Graficar errores con signo del banco corto para representaciones reajustadas y congeladas de las
# cinco formas..
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
fig, ax = plt.subplots(figsize=(6.4, 3.8))
residence.set_index("Modelo")[
    ["Error con signo de reajuste (pp)", "Error con signo congelado (pp)"]
].plot.barh(ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set(xlabel="Error de banco\n(pp)", ylabel="", title="Errores del banco corto")
ax.tick_params(axis="x", rotation=0)
ax.legend(["Reajustado", "Congelado"], fontsize=18, ncol=2, **legend_style)
fig.subplots_adjust(left=0.4, right=0.96, bottom=0.23, top=0.72)


Las barras comparan cada modelo con la misma referencia del banco corto. Reajustar permite identificar otros parámetros; congelar conserva el vector nominal y cambia solo la geometría de proyección de producción. Barras similares son un resultado, no equivalencia entre procedimientos. Magnitud y sentido del error diagnostican aproximación del simulador, sin establecer desempeño en una campaña de flotación independiente.

## Resultados

**Respuesta:** En el banco corto, los errores con signo de Kelsall son <!-- result:Error de Kelsall reajustado en banco corto (pp) -->0<!-- /result --> pp tras reajuste y <!-- result:Error de Kelsall congelado en banco corto (pp) -->0<!-- /result --> pp con parámetros nominales congelados. Tabla y figura comparan las cinco formas con la misma referencia modificada. El ajuste nominal no responde por sí solo esta pregunta; reajuste y transferencia permanecen identificados por separado.

Resumir errores de Kelsall con residencia modificada y parámetros reajustados o congelados; la tabla anterior conserva cinco formas.

In [ ]:
# Resumir las salidas calculadas que sostienen la interpretación de esta lección.
kinetic_by_name = {model["id"]: model for model in fits["models"]}
answer_values = {
    "Recuperación global del banco corto (%)": short.metrics["rougher_recovery_pct"],
    "Recuperación verdadera del banco corto (%)": short_fits["bank"]["exact_true_flotation_pct"],
    "RMSE batch de primer orden (pp)": kinetic_by_name["first_order"]["rmse_pct"],
    "RMSE batch de Kelsall (pp)": kinetic_by_name["kelsall"]["rmse_pct"],
    "Error de banco de primer orden (pp)": abs(kinetic_by_name["first_order"]["lumping_error_pct"]),
    "Error de banco de Kelsall (pp)": abs(kinetic_by_name["kelsall"]["lumping_error_pct"]),
}
short_kelsall = short_by_id["kelsall"]
frozen_kelsall = next((row for row in residence_rows if row["Modelo"] == "kelsall"))
answer_values.update(
    {
        "Error de Kelsall reajustado en banco corto (pp)": short_kelsall["lumping_error_pct"],
        "Error de Kelsall congelado en banco corto (pp)": frozen_kelsall[
            "Error con signo congelado (pp)"
        ],
    }
)
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

La comparación bibliográfica muestra tanto la capacidad como el límite: varias formas cinéticas ajustan una curva sintética, pero sus predicciones de banco divergen. OreFlow informa la diferencia en vez de presentar la animación como dinámica de planta.

## Referencias

- https://doi.org/10.1016/S0301-7516(99)00069-1
- https://doi.org/10.5277/ppmp170128
- https://doi.org/10.1016/j.mineng.2025.109246
- https://doi.org/10.1016/0301-7516(81)90019-3